In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import os

KERNELS = [3, 5, 11, 29]
arquivos = ["alunos", "avatares"]
src = "../img"
dst = "./filtros_passa_baixa"
filtros_lista = ["Média", "Gaussiano", "Mediana", "Bilateral"]

for f in ["media", "gaussiana", "mediana", "bilateral"]:
    for arq in arquivos:
        os.makedirs(f"{dst}/{f}/{arq}", exist_ok=True)

def gerar_gaussiano_severo(img):
    mean, sigma = 0, 80
    gauss = np.random.normal(mean, sigma, img.shape).astype('float32')
    img_ruidosa = cv.add(img.astype('float32'), gauss)
    return np.clip(img_ruidosa, 0, 255).astype('uint8')

def gerar_sal_pimenta_severo(img):
    prob = 0.2
    img_ruidosa = np.copy(img)
    rdn = np.random.random(img.shape[:2])
    img_ruidosa[rdn < (prob / 2)] = [255, 255, 255]
    img_ruidosa[rdn > (1 - prob / 2)] = [0, 0, 0]
    return img_ruidosa

def redimensionar_para_stack(lista_imgs):
    if not lista_imgs: return []
    h_min = min(img.shape[0] for img in lista_imgs)
    return [cv.resize(img, (int(img.shape[1] * h_min / img.shape[0]), h_min)) for img in lista_imgs]

def calcular_metricas_filtros(original, ruidosa, title, modo_ruido, tipo_filtro):
    plt.figure(figsize=(18, 6))
    plt.suptitle(f"Filtro {tipo_filtro} - {title} ({modo_ruido})", fontsize=16)
    
    for i, k in enumerate(KERNELS):
        if tipo_filtro == "Mediana":
            img_dst = cv.medianBlur(ruidosa, k)
            nome_legenda = f"median_blur {k}x{k}"
            pasta = "mediana"
        elif tipo_filtro == "Bilateral":
            img_dst = cv.bilateralFilter(ruidosa, k, k * 2, k / 2)
            nome_legenda = f"bilateral_blur {k}x{k}"
            pasta = "bilateral"
        elif tipo_filtro == "Gaussiano":
            img_dst = cv.GaussianBlur(ruidosa, (k, k), 0)
            nome_legenda = f"gaussian_blur {k}x{k}"
            pasta = "gaussiana"
        else:
            img_dst = cv.blur(ruidosa, (k, k))
            nome_legenda = f"blur {k}x{k}"
            pasta = "media"

        dst_file_name = f"{dst}/{pasta}/{title}/{modo_ruido}_{k}x{k}.jpg"
        cv.imwrite(dst_file_name, cv.cvtColor(img_dst, cv.COLOR_RGB2BGR))

        psnr_v = cv.PSNR(original, img_dst)

        plt.subplot(1, len(KERNELS), i + 1)
        plt.imshow(img_dst)
        plt.title(f'{nome_legenda}\nPSNR: {psnr_v:.2f}dB', fontsize=10)
        plt.axis('off')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

imagens_originais = []
imagens_gauss = []
imagens_sp = []

for file in arquivos:
    image_path = f'{src}/{file}.png'
    img = cv.imread(image_path)
    if img is None: continue
    
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    imagens_originais.append(img)
    imagens_gauss.append(gerar_gaussiano_severo(img))
    imagens_sp.append(gerar_sal_pimenta_severo(img))

if imagens_gauss:
    plt.figure(figsize=(12, 6))
    plt.imshow(np.hstack(redimensionar_para_stack(imagens_gauss)))
    plt.title("Imagens de Teste 1: Ruído Gaussiano Severo", fontsize=14)
    plt.axis('off')
    plt.show()

    plt.figure(figsize=(12, 6))
    plt.imshow(np.hstack(redimensionar_para_stack(imagens_sp)))
    plt.title("Imagens de Teste 2: Ruído Sal e Pimenta Severo", fontsize=14)
    plt.axis('off')
    plt.show()

    for idx, img_r in enumerate(imagens_gauss):
        for nome_f in filtros_lista:
            calcular_metricas_filtros(imagens_originais[idx], img_r, arquivos[idx], "Gauss", nome_f)

    for idx, img_r in enumerate(imagens_sp):
        for nome_f in filtros_lista:
            calcular_metricas_filtros(imagens_originais[idx], img_r, arquivos[idx], "SalPimenta", nome_f)